In [ ]:
# =====================================
# SparkBody 完整分析 + 動畫 - 單 Cell
# IEEE TAC Affective Intelligence & Art Design
# =====================================

import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.animation as animation
import seaborn as sns
from pathlib import Path

# =========================
# ⚙️ 路徑設定 (只需改這裡)
# =========================
TSV_PATH  = r"C:\Users\user\posefireworks\analysis_lab\raw_data\SparkBodyDaTa V.2 -  Sheet1 (1).tsv"
FFMPEG_EXE = r"C:\Users\user\posefireworks\analysis_lab\ffmpeg.exe"
PLOTS_DIR  = Path("plots")
PLOTS_DIR.mkdir(exist_ok=True)

LMA_COLS   = ["shape_n", "weight_n", "flow_n", "kt"]
ACTIVITIES = ["Heart", "Victory", "Fireworks_Explosion", "Gull_Flap", "Dance_Continuous"]
COLORS     = ["#E63946", "#457B9D", "#F4A261", "#2A9D8F", "#8338EC"]
ACT_COLOR  = dict(zip(ACTIVITIES, COLORS))

# =========================
# 1️⃣ 讀取資料
# =========================
with open(TSV_PATH, encoding="utf-8") as f:
    raw = f.read()

tsv_lines = [l for l in raw.split("\n") if "\t" in l]
df_raw = pd.read_csv(
    io.StringIO("\n".join(tsv_lines)),
    sep="\t", header=None,
    names=["sessionId","userId","timestamp","mode","activity",
           "shape_n","weight_n","flow_n","kt","baselineReady",
           "lh_x","lh_y","rh_x","rh_y","ls_x","ls_y","rs_x","rs_y","note"]
)
df_raw["baselineReady"] = df_raw["baselineReady"].astype(str).str.upper() == "TRUE"
df_raw["timestamp"]    = pd.to_datetime(df_raw["timestamp"], errors="coerce")
for c in LMA_COLS:
    df_raw[c] = pd.to_numeric(df_raw[c], errors="coerce")
coord_cols = ["lh_x","lh_y","rh_x","rh_y","ls_x","ls_y","rs_x","rs_y"]
for c in coord_cols:
    df_raw[c] = pd.to_numeric(df_raw[c], errors="coerce")

n_total    = len(df_raw)
n_excluded = (~df_raw["baselineReady"]).sum()
df         = df_raw[df_raw["baselineReady"]].dropna(subset=coord_cols).copy().reset_index(drop=True)
n_valid    = len(df)
n_sessions = df["sessionId"].nunique()

print(f"原始總筆數    : {n_total}")
print(f"排除 FALSE    : {n_excluded} 筆")
print(f"分析用筆數    : {n_valid} 筆")
print(f"Session 數    : {n_sessions} 人")
print(f"Activity 種類 : {sorted(df['activity'].unique())}")

# =========================
# 2️⃣ 論文用統計 mean ± SD
# =========================
df_act = df[df["activity"].isin(ACTIVITIES)].copy()

print("\n=== Overall LMA mean ± SD ===")
for col in LMA_COLS:
    print(f"  {col:<12}: {df[col].mean():.3f} ± {df[col].std():.3f}")

print("\n=== Per-activity kt mean ± SD ===")
for act in ACTIVITIES:
    sub = df[df["activity"] == act]["kt"]
    if len(sub) > 0:
        print(f"  {act:<25}: {sub.mean():.3f} ± {sub.std():.3f}  (n={len(sub)})")

print("\n=== Per-session LMA mean ===")
print(df.groupby("sessionId")[LMA_COLS].mean().round(3))

# =========================
# 3️⃣ 圖1：盒鬚圖 LMA × Activity
# =========================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("LMA Indicators by Activity", fontsize=15, fontweight="bold")
for ax, col in zip(axes.flat, LMA_COLS):
    data = [df_act[df_act["activity"] == a][col].dropna().values for a in ACTIVITIES]
    bp = ax.boxplot(data, patch_artist=True, medianprops=dict(color="white", linewidth=2))
    for patch, color in zip(bp["boxes"], COLORS):
        patch.set_facecolor(color); patch.set_alpha(0.8)
    ax.set_xticks(range(1, len(ACTIVITIES)+1))
    ax.set_xticklabels([a.replace("_", "\n") for a in ACTIVITIES], fontsize=9)
    ax.set_title(col, fontsize=12, fontweight="bold")
    ax.set_ylabel("Normalized Value"); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "fig1_lma_boxplot.png", dpi=300, bbox_inches="tight")
plt.show(); print("✅ fig1 saved")

# =========================
# 4️⃣ 圖2：熱力圖 Activity × LMA
# =========================
heatmap_data = df_act.groupby("activity")[LMA_COLS].mean().reindex(ACTIVITIES)
fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0, vmax=1, linewidths=0.5, ax=ax,
            cbar_kws={"label": "Mean Normalized Value"})
ax.set_title("Mean LMA Indicators per Activity", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "fig2_lma_heatmap.png", dpi=300, bbox_inches="tight")
plt.show(); print("✅ fig2 saved")

# =========================
# 5️⃣ 圖3：雷達圖 Session × LMA
# =========================
session_means = df.groupby("sessionId")[LMA_COLS].mean()
N      = len(LMA_COLS)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
cmap = plt.cm.get_cmap("tab10", len(session_means))
for i, (sid, row) in enumerate(session_means.iterrows()):
    values = row[LMA_COLS].tolist() + [row[LMA_COLS[0]]]
    ax.plot(angles, values, linewidth=2, color=cmap(i), label=sid[:12])
    ax.fill(angles, values, alpha=0.1, color=cmap(i))
ax.set_xticks(angles[:-1]); ax.set_xticklabels(LMA_COLS, fontsize=12)
ax.set_title("LMA Profile per Session", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=8)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "fig3_lma_radar.png", dpi=300, bbox_inches="tight")
plt.show(); print("✅ fig3 saved")

# =========================
# 6️⃣ 圖4：kt 時序圖 per session
# =========================
sessions = df["sessionId"].unique()
fig, axes = plt.subplots(len(sessions), 1, figsize=(14, 3*len(sessions)))
if len(sessions) == 1: axes = [axes]
for ax, sid in zip(axes, sessions):
    sdf = df[df["sessionId"] == sid].reset_index(drop=True)
    ax.plot(sdf.index, sdf["kt"], color="#457B9D", linewidth=2)
    ax.fill_between(sdf.index, sdf["kt"], alpha=0.15, color="#457B9D")
    for idx, row in sdf.iterrows():
        if row["activity"] in ACT_COLOR:
            ax.axvline(idx, color=ACT_COLOR[row["activity"]], alpha=0.3, linewidth=1)
    ax.set_title(f"{sid}", fontsize=9); ax.set_ylim(0, 1.05)
    ax.set_ylabel("kt"); ax.grid(alpha=0.2)
patches = [mpatches.Patch(color=c, label=a) for a, c in ACT_COLOR.items()]
fig.legend(handles=patches, loc="lower center", ncol=5, fontsize=9, bbox_to_anchor=(0.5, -0.01))
fig.suptitle("Kinetic Tone (kt) Time Series per Session", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "fig4_kt_timeseries.png", dpi=300, bbox_inches="tight")
plt.show(); print("✅ fig4 saved")

# =========================
# 7️⃣ 動畫 MP4 — 全部 session
# =========================
def export_session_mp4(sid):
    vsdf = df[df["sessionId"] == sid].copy().reset_index(drop=True)
    if vsdf.empty:
        print(f"⚠️ 無資料: {sid}"); return

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.set_xlim(1, 0); ax.set_ylim(1, 0)
    ax.set_title(f"Session: {sid}", fontsize=11)
    line_l, = ax.plot([], [], "g-o", lw=4, label="Left Arm")
    line_r, = ax.plot([], [], "r-o", lw=4, label="Right Arm")
    line_s, = ax.plot([], [], "b-",  lw=4, alpha=0.5, label="Shoulders")
    act_text = ax.text(0.05, 0.05, "", transform=ax.transAxes, fontsize=9, color="gray")
    ax.legend(loc="upper right", fontsize=8)

    def update(frame):
        row = vsdf.iloc[frame]
        line_l.set_data([row.ls_x, row.lh_x], [row.ls_y, row.lh_y])
        line_r.set_data([row.rs_x, row.rh_x], [row.rs_y, row.rh_y])
        line_s.set_data([row.ls_x, row.rs_x], [row.ls_y, row.rs_y])
        act_text.set_text(f"{row.activity}  kt={row.kt:.2f}")
        return line_l, line_r, line_s, act_text

    ani = animation.FuncAnimation(fig, update, frames=len(vsdf), interval=200, blit=True)
    try:
        writer = animation.FFMpegWriter(fps=3)
        writer.executable = FFMPEG_EXE
        save_path = PLOTS_DIR / f"{sid}.mp4"
        ani.save(save_path, writer=writer)
        print(f"🎬 {sid}.mp4 done")
    except Exception as e:
        print(f"❌ {sid} 失敗: {e}")
    plt.close(fig)

print("\n🎬 開始產出所有 session MP4...")
for sid in df["sessionId"].unique():
    export_session_mp4(sid)

# =========================
# ✅ 完成摘要
# =========================
print("\n" + "="*50)
print("📄 論文 Methods 數字（直接貼用）")
print("="*50)
print(f"A total of {n_total} records were collected from {n_sessions} participants.")
print(f"{n_excluded} pre-baseline records were excluded, yielding {n_valid} valid observations.")
print()
for col in LMA_COLS:
    print(f"  {col}: M={df[col].mean():.3f}, SD={df[col].std():.3f}")
print(f"\n✅ 所有輸出存於: {PLOTS_DIR.resolve()}")
